# V6 Online Adaptive State Replay

> Self-contained Colab notebook. No GitHub clone is required.
>
> Paths assume Google Drive layout used throughout this project.

## Goal

Simulate production-style online adaptive serving:

```text
prior events -> update online state -> query -> V4 retrieval -> V5 reranking -> Top-K
```

V6 trains no model. It loads V4 and V5 artifacts and compares V3, V4, and V4+V5.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import importlib.util, subprocess, sys
required = ['pyarrow', 'torch', 'pandas', 'numpy']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('packages ready')

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict, deque
import json, math, random
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import torch
from torch import nn
from torch.nn import functional as F

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
@dataclass
class V6Config:
    adaptive_root: str = '/content/drive/MyDrive/recsys/data/gold/adaptive/v1'
    base_gold_root: str = '/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab'
    v3_dir: str = '/content/drive/MyDrive/recsys/artifacts/two_tower/baseline_v3'
    v4_dir: str = '/content/recsys_work/artifacts/adaptive/v4_sequence_retrieval'
    v5_dir: str = '/content/recsys_work/artifacts/adaptive/v5_adaptive_ranker'
    output_dir: str = '/content/recsys_work/artifacts/adaptive/v6_online_replay'
    max_replay_queries: int | None = 5000
    candidate_pool_split: str = 'validation'
    retrieval_candidates_per_query: int = 1000
    top_ks: tuple[int, ...] = (10,50,100)
    sequence_max_len: int = 30
    save_large_cache_to_drive: bool = False
    max_report_rows: int = 5000
    local_cache_dir: str = '/content/recsys_work/adaptive_cache/v6'

CONFIG=V6Config(); ADAPTIVE_ROOT=Path(CONFIG.adaptive_root); BASE_GOLD_ROOT=Path(CONFIG.base_gold_root); OUTPUT_DIR=Path(CONFIG.output_dir); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
LOCAL_CACHE_DIR=Path(CONFIG.local_cache_dir); LOCAL_CACHE_DIR.mkdir(parents=True,exist_ok=True)
print(json.dumps(asdict(CONFIG),indent=2,default=str))

In [ ]:
# V6 intentionally reuses the same helper/model code from V4/V5 notebooks.
# To keep this notebook readable, run notebooks 12 and 13 first, then use their saved metrics/artifacts here.
def parquet_files(path):
    path=Path(path); files=sorted(p for p in path.rglob('*.parquet') if p.is_file() and not p.name.startswith('.'))
    if not files: raise FileNotFoundError(path)
    return files

def read_parquet(path, columns=None, max_rows=None, filter_expr=None):
    dataset=ds.dataset([str(p) for p in parquet_files(path)],format='parquet')
    return (dataset.to_table(columns=columns,filter=filter_expr) if max_rows is None else dataset.head(max_rows,columns=columns,filter=filter_expr)).to_pandas()

def save_json(path,payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w') as f: json.dump(payload,f,indent=2,sort_keys=True,default=str)

In [ ]:
def as_sequence(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return []


def as_of_time_ms(frame: pd.DataFrame) -> np.ndarray:
    return (pd.to_datetime(frame['as_of_time']).astype('int64') // 1_000_000).to_numpy(dtype='int64')


def sanitize_point_in_time_sequences(frame: pd.DataFrame) -> pd.DataFrame:
    """Drop any sequence events that are not strictly before the target as_of_time.

    This is intentionally conservative and protects training/evaluation even if the
    adaptive data builder attached a same-second history row ambiguously.
    """
    frame = frame.copy()
    boundaries = as_of_time_ms(frame)
    sequence_cols = [
        'recent_video_ids', 'recent_target_classes', 'recent_engagement_strengths',
        'recent_long_views', 'recent_likes', 'recent_hates', 'recent_clicks', 'recent_time_ms',
    ]
    cleaned = {col: [] for col in sequence_cols if col in frame.columns}
    cleaned_counts = []
    dropped_rows = 0
    dropped_events = 0
    for (_, row), boundary in zip(frame.iterrows(), boundaries):
        times = as_sequence(row.get('recent_time_ms'))
        keep = [i for i, t in enumerate(times) if t is not None and not pd.isna(t) and float(t) < float(boundary)]
        if len(keep) != len(times):
            dropped_rows += 1
            dropped_events += len(times) - len(keep)
        for col in cleaned:
            values = as_sequence(row.get(col))
            if len(values) == len(times):
                cleaned[col].append([values[i] for i in keep])
            else:
                cleaned[col].append(values[-len(keep):] if keep else [])
        cleaned_counts.append(len(keep))
    for col, values in cleaned.items():
        frame[col] = values
    frame['recent_event_count'] = cleaned_counts
    frame['current_time_ms'] = boundaries
    print(f'point-in-time sanitize: rows_with_drops={dropped_rows:,} dropped_events={dropped_events:,}')
    return frame

In [ ]:
# Load saved metric references first. Replay cells below produce fresh V4/V5 deltas on the online query slice.
v3_metrics_path=Path(CONFIG.v3_dir)/'metrics.json'
v4_metrics_path=Path(CONFIG.v4_dir)/'metrics.json'
v5_metrics_path=Path(CONFIG.v5_dir)/'metrics.json'
for name,path in [('v3',v3_metrics_path),('v4',v4_metrics_path),('v5',v5_metrics_path)]:
    print(name, path, path.exists())
    if path.exists():
        payload=json.loads(path.read_text())
        preview=payload.get('candidate_validation', payload.get('ranking_comparison', payload))
        print(json.dumps(preview, indent=2, default=str)[:2000])

In [ ]:
# Online replay input. If event_stream/serving_queries were not materialized, derive a lean replay view
# from validation sequence_examples so V6 can run with the existing adaptive dataset.
event_cols=['event_id','user_id','video_id','event_ts','time_ms','session_id','target_class','engagement_strength','long_view','is_like','is_hate','is_click']
query_cols=['split','example_id','user_id','video_id','session_id','as_of_time','target_class','is_positive','engagement_strength','recent_event_count','session_recent_event_count']

def existing_cols(path: Path) -> list[str]:
    files=parquet_files(path)
    schema=pq.ParquetFile(files[0]).schema_arrow
    return schema.names

if (ADAPTIVE_ROOT/'event_stream').exists():
    events=read_parquet(ADAPTIVE_ROOT/'event_stream', event_cols).sort_values('time_ms').reset_index(drop=True)
else:
    print('event_stream not found; deriving events from sequence_examples/validation')
    seq_path=ADAPTIVE_ROOT/'sequence_examples/validation'
    available=set(existing_cols(seq_path))
    source_cols=[c for c in ['example_id','user_id','video_id','as_of_time','current_time_ms','session_id','target_class','engagement_strength','long_view','is_like','is_hate','is_click'] if c in available]
    events=read_parquet(seq_path, source_cols, CONFIG.max_replay_queries).copy()
    events['event_id']=events.get('example_id', pd.Series(np.arange(len(events))))
    events['event_ts']=events['as_of_time']
    if 'current_time_ms' in events.columns:
        events['time_ms']=pd.to_numeric(events['current_time_ms'], errors='coerce')
    else:
        events['time_ms']=(pd.to_datetime(events['as_of_time']).astype('int64')//1_000_000)
    for col in event_cols:
        if col not in events.columns:
            events[col]=0
    events=events[event_cols].sort_values('time_ms').reset_index(drop=True)

if (ADAPTIVE_ROOT/'serving_queries/validation').exists():
    queries=read_parquet(ADAPTIVE_ROOT/'serving_queries/validation', query_cols, CONFIG.max_replay_queries).sort_values('as_of_time').reset_index(drop=True)
else:
    print('serving_queries/validation not found; deriving queries from sequence_examples/validation')
    seq_path=ADAPTIVE_ROOT/'sequence_examples/validation'
    available=set(existing_cols(seq_path))
    source_cols=[c for c in query_cols if c in available]
    queries=read_parquet(seq_path, source_cols, CONFIG.max_replay_queries, ds.field('target_class')=='STRONG_POSITIVE').copy()
    queries['split']='validation'
    for col in query_cols:
        if col not in queries.columns:
            queries[col]=0
    queries=queries[query_cols].sort_values('as_of_time').reset_index(drop=True)

queries['query_time_ms']=(pd.to_datetime(queries['as_of_time']).astype('int64')//1_000_000).astype('int64')
print('events/queries:', events.shape, queries.shape)

In [ ]:
# This lightweight state replay validates timestamp ordering and produces state snapshots.
# Full V4/V5 scoring should be run after notebooks 12/13 complete; this cell prepares the online state contract.
recent_by_user=defaultdict(lambda: deque(maxlen=CONFIG.sequence_max_len))
state_rows=[]
event_ptr=0
for _,q in queries.iterrows():
    qt=int(q['query_time_ms'])
    while event_ptr < len(events) and int(events.iloc[event_ptr]['time_ms']) < qt:
        e=events.iloc[event_ptr]
        recent_by_user[int(e['user_id'])].append({
            'video_id': int(e['video_id']), 'target_class': e['target_class'], 'engagement_strength': float(e['engagement_strength'] or 0),
            'long_view': int(e['long_view'] or 0), 'is_like': int(e['is_like'] or 0), 'is_hate': int(e['is_hate'] or 0), 'is_click': int(e['is_click'] or 0), 'time_ms': int(e['time_ms'])
        })
        event_ptr += 1
    hist=list(recent_by_user[int(q['user_id'])])
    if any(h['time_ms'] >= qt for h in hist):
        raise AssertionError('Online state leakage: query saw a future/current event')
    state_rows.append({'example_id':q['example_id'],'user_id':int(q['user_id']),'query_time_ms':qt,'history_len':len(hist),'recent_video_ids':[h['video_id'] for h in hist],'recent_target_classes':[h['target_class'] for h in hist]})
state_df=pd.DataFrame(state_rows)
print(state_df.head())
print('history length summary')
print(state_df['history_len'].describe())
snapshot_path = (OUTPUT_DIR if CONFIG.save_large_cache_to_drive else LOCAL_CACHE_DIR) / 'online_state_snapshots.parquet'
state_df.to_parquet(snapshot_path, index=False)
print('state snapshots written to:', snapshot_path)

## V4/V5 Exact Matrix Scoring On Replay Queries

This section rebuilds the deterministic V4 vocabularies, loads V4 and V5 artifacts, scores the replay query slice with exact matrix search, and compares:

```text
V4 = V4 candidates ordered by retrieval score
V4 + V5 = same candidates reranked by the V5 rank score
```

V3 is included as a saved static-reference artifact because its candidate universe may differ from the adaptive replay slice.

In [ ]:

USER_CATEGORICAL=('user_active_degree',); ITEM_CATEGORICAL=('video_type','upload_type'); USER_ID_FEATURES=('user_id',); ITEM_ID_FEATURES=('video_id',)
USER_NUMERIC=('is_lowactive_period','is_live_streamer','is_video_author','follow_user_num','fans_user_num','friend_user_num','register_days',*[f'onehot_feat{i}' for i in range(18)],'user_hist_events','user_hist_long_view_rate','user_hist_like_rate','user_hist_comment_rate','user_hist_forward_rate','user_hist_follow_rate','user_hist_hate_rate','user_hist_avg_watch_ratio','user_hist_avg_play_time_sec','session_event_index','session_elapsed_sec','session_prior_long_view_rate','session_prior_like_rate','session_prior_hate_rate','session_prior_avg_watch_ratio','session_vs_user_long_view_delta','session_vs_user_watch_ratio_delta','event_hour','event_dayofweek','tab','is_rand')
ITEM_NUMERIC=('video_duration_sec','aspect_ratio','visible_status','music_type','upload_age_days_at_event','item_hist_events','item_hist_long_view_rate','item_hist_like_rate','item_hist_hate_rate','item_hist_avg_watch_ratio')
SEQUENCE_COLUMNS=('recent_video_ids','recent_target_classes','recent_engagement_strengths','recent_long_views','recent_likes','recent_hates','recent_clicks','recent_time_ms','recent_event_count','seconds_since_last_event')
TARGET_COLUMNS=('target_class','is_positive','is_observed_negative','is_ambiguous','long_view','is_like','is_follow','is_hate','engagement_strength')
BASE_COLUMNS=('example_id','user_id','video_id','as_of_time','current_time_ms')
INPUT_COLUMNS=list(dict.fromkeys([*BASE_COLUMNS,*TARGET_COLUMNS,*USER_ID_FEATURES,*USER_CATEGORICAL,*USER_NUMERIC,*ITEM_ID_FEATURES,*ITEM_CATEGORICAL,*ITEM_NUMERIC,*SEQUENCE_COLUMNS]))

class Vocabulary:
    def __init__(self, name, values, index=None):
        self.name=name
        if index is None:
            clean=pd.Series(list(values)).dropna().drop_duplicates().tolist(); clean=sorted(clean)
            self.index={v:i+1 for i,v in enumerate(clean)}
        else:
            self.index=dict(index)
        self.size=max(self.index.values(), default=0)+1
    def encode_many(self, values):
        return np.asarray([self.index.get(v,0) for v in values], dtype=np.int64)

def load_parquet_vocabulary(path, feature):
    df=read_parquet(path)
    return Vocabulary(feature, [], dict(zip(df[feature].tolist(), df[f'{feature}_idx'].astype(int).tolist())))

class NumericalPreprocessor:
    def __init__(self, path):
        payload=json.loads(Path(path).read_text()); self.stats=payload.get('features', payload)
    def transform(self, frame, features):
        outs=[]
        for feat in features:
            vals=pd.to_numeric(frame[feat], errors='coerce').astype('float32') if feat in frame.columns else pd.Series(np.zeros(len(frame)), dtype='float32')
            st=self.stats.get(feat)
            if st:
                vals=vals.fillna(float(st.get('mean',0) or 0)); lo=st.get('p01',st.get('min')); hi=st.get('p99',st.get('max'))
                if lo is not None and hi is not None: vals=vals.clip(float(lo),float(hi))
                std=float(st.get('stddev',0) or 0); mean=float(st.get('mean',0) or 0); vals=(vals-mean)/std if std>1e-6 else vals*0
            outs.append(np.nan_to_num(vals.to_numpy('float32'), nan=0, posinf=0, neginf=0))
        return np.stack(outs, axis=1).astype('float32') if outs else np.zeros((len(frame),0),'float32')

def build_vocabs_for_v4(v4_cfg, item_pool):
    user_cols=['user_id']
    train_users=read_parquet(ADAPTIVE_ROOT/'sequence_examples/train', user_cols, v4_cfg.get('max_train_examples'), ds.field('target_class')=='STRONG_POSITIVE')
    return {
        'user_active_degree':load_parquet_vocabulary(Path(CONFIG.base_gold_root)/'vocabularies/user_active_degree','user_active_degree'),
        'video_type':load_parquet_vocabulary(Path(CONFIG.base_gold_root)/'vocabularies/video_type','video_type'),
        'upload_type':load_parquet_vocabulary(Path(CONFIG.base_gold_root)/'vocabularies/upload_type','upload_type'),
        'user_id':Vocabulary('user_id',train_users['user_id'].tolist()),
        'video_id':Vocabulary('video_id',item_pool['video_id'].tolist()),
        'event_type':Vocabulary('event_type',['AMBIGUOUS_WEAK','OBSERVED_NEGATIVE','STRONG_POSITIVE']),
    }

def pad_num(v,max_len,dtype='float32'):
    v=[] if not isinstance(v,(list,tuple,np.ndarray)) else list(v)[-max_len:]
    out=np.zeros(max_len,dtype=dtype)
    if v: out[-len(v):]=np.asarray(v,dtype=dtype)
    return out

def pad_enc(v,vocab,max_len):
    v=[] if not isinstance(v,(list,tuple,np.ndarray)) else list(v)[-max_len:]
    out=np.zeros(max_len,dtype=np.int64)
    if v: out[-len(v):]=vocab.encode_many(v)[-max_len:]
    return out

def encode_frame(frame, vocabs, numeric, max_len):
    b={}
    for f in USER_ID_FEATURES+USER_CATEGORICAL+ITEM_ID_FEATURES+ITEM_CATEGORICAL:
        b[f]=torch.as_tensor(vocabs[f].encode_many(frame[f].tolist()), dtype=torch.long)
    b['user_numeric']=torch.as_tensor(numeric.transform(frame, USER_NUMERIC), dtype=torch.float32)
    b['item_numeric']=torch.as_tensor(numeric.transform(frame, ITEM_NUMERIC), dtype=torch.float32)
    b['seq_video_id']=torch.as_tensor(np.stack([pad_enc(v,vocabs['video_id'],max_len) for v in frame['recent_video_ids']]), dtype=torch.long)
    b['seq_event_type']=torch.as_tensor(np.stack([pad_enc(v,vocabs['event_type'],max_len) for v in frame['recent_target_classes']]), dtype=torch.long)
    sig=np.stack([np.stack([pad_num(v,max_len) for v in frame[c]]) for c in ['recent_engagement_strengths','recent_long_views','recent_likes','recent_hates','recent_clicks']], axis=-1)
    b['seq_signals']=torch.as_tensor(sig, dtype=torch.float32)
    cur=pd.to_numeric(frame['current_time_ms'],errors='coerce').fillna(0).to_numpy('float64')
    rec=np.stack([pad_num(v,max_len,'float64') for v in frame['recent_time_ms']])
    age=np.maximum((cur[:,None]-rec)/1000,0); age[rec<=0]=0
    b['seq_time_bucket']=torch.as_tensor(np.clip(np.floor(np.log1p(age)/np.log(2)),0,31).astype('int64'))
    b['seq_padding_mask']=b['seq_video_id'].eq(0)
    return b

def move(batch,device):
    return {k:v.to(device) if torch.is_tensor(v) else v for k,v in batch.items()}

def minimal_user_stub(source_row, n):
    row=source_row
    data={c: row.get(c,0) for c in USER_NUMERIC}
    data.update({'user_id':row['user_id'],'user_active_degree':row.get('user_active_degree'),'recent_video_ids':[row.get('recent_video_ids',[]) for _ in range(n)],'recent_target_classes':[row.get('recent_target_classes',[]) for _ in range(n)],'recent_engagement_strengths':[row.get('recent_engagement_strengths',[]) for _ in range(n)],'recent_long_views':[row.get('recent_long_views',[]) for _ in range(n)],'recent_likes':[row.get('recent_likes',[]) for _ in range(n)],'recent_hates':[row.get('recent_hates',[]) for _ in range(n)],'recent_clicks':[row.get('recent_clicks',[]) for _ in range(n)],'recent_time_ms':[row.get('recent_time_ms',[]) for _ in range(n)],'current_time_ms':row.get('current_time_ms',0)})
    return data

def mlp(input_dim, hidden_dims, output_dim, dropout):
    layers=[]; prev=input_dim
    for h in hidden_dims:
        layers += [nn.Linear(prev,h), nn.ReLU(), nn.Dropout(dropout)]; prev=h
    layers.append(nn.Linear(prev,output_dim)); return nn.Sequential(*layers)

class V4TransformerRetrieval(nn.Module):
    def __init__(self, vocab_sizes, cfg):
        super().__init__(); rd=cfg['retrieval_dim']; td=cfg['transformer_dim']
        self.user_id_emb=nn.Embedding(vocab_sizes['user_id'],8,padding_idx=0); self.user_active_emb=nn.Embedding(vocab_sizes['user_active_degree'],4,padding_idx=0)
        self.video_id_emb=nn.Embedding(vocab_sizes['video_id'],16,padding_idx=0); self.video_type_emb=nn.Embedding(vocab_sizes['video_type'],3,padding_idx=0); self.upload_type_emb=nn.Embedding(vocab_sizes['upload_type'],8,padding_idx=0)
        self.seq_video_emb=nn.Embedding(vocab_sizes['video_id'],td,padding_idx=0); self.seq_event_emb=nn.Embedding(vocab_sizes['event_type'],32,padding_idx=0); self.seq_event_proj=nn.Linear(32,td); self.seq_signal_proj=nn.Linear(5,td); self.seq_time_emb=nn.Embedding(32,td)
        layer=nn.TransformerEncoderLayer(td,cfg['transformer_heads'],td*4,cfg['transformer_dropout'],batch_first=True,activation='gelu')
        self.transformer=nn.TransformerEncoder(layer,cfg['transformer_layers'])
        self.short_proj=nn.Sequential(nn.LayerNorm(td),nn.Linear(td,rd),nn.ReLU())
        self.long_user=mlp(8+4+len(USER_NUMERIC),tuple(cfg['user_hidden_dims']),rd,cfg['dropout'])
        self.item_tower=mlp(16+3+8+len(ITEM_NUMERIC),tuple(cfg['item_hidden_dims']),rd,cfg['dropout'])
        self.gate=nn.Sequential(nn.Linear(rd*2,rd),nn.Sigmoid())
    def encode_sequence(self,b):
        tok=self.seq_video_emb(b['seq_video_id'])+self.seq_event_proj(self.seq_event_emb(b['seq_event_type']))+self.seq_signal_proj(b['seq_signals'])+self.seq_time_emb(b['seq_time_bucket'].clamp(0,31))
        mask=b['seq_padding_mask']; enc=self.transformer(tok,src_key_padding_mask=mask); valid=(~mask).float().unsqueeze(-1)
        return self.short_proj((enc*valid).sum(1)/valid.sum(1).clamp_min(1))
    def encode_user(self,b):
        long=self.long_user(torch.cat([self.user_id_emb(b['user_id']),self.user_active_emb(b['user_active_degree']),b['user_numeric']],1)); short=self.encode_sequence(b); g=self.gate(torch.cat([long,short],1)); return F.normalize(g*short+(1-g)*long,dim=1)
    def encode_item(self,b):
        return F.normalize(self.item_tower(torch.cat([self.video_id_emb(b['video_id']),self.video_type_emb(b['video_type']),self.upload_type_emb(b['upload_type']),b['item_numeric']],1)),dim=1)

class V5Ranker(nn.Module):
    def __init__(self, retrieval_dim, hidden_dims, dropout):
        super().__init__(); self.net=mlp(retrieval_dim*4+1, hidden_dims, 1, dropout)
    def forward(self, user_emb, item_emb, v4_score):
        return self.net(torch.cat([user_emb,item_emb,user_emb*item_emb,torch.abs(user_emb-item_emb),v4_score[:,None]],1)).squeeze(1)


In [ ]:
# Load V4/V5 artifacts and score the replay query slice.
v4_path=Path(CONFIG.v4_dir)/'best_model.pt'
v5_path=Path(CONFIG.v5_dir)/'best_model.pt'
print('V4 checkpoint:', v4_path, 'exists=', v4_path.exists())
print('V5 checkpoint:', v5_path, 'exists=', v5_path.exists())
assert v4_path.exists(), f'Missing V4 checkpoint: {v4_path}. Run the V4 section first.'
assert v5_path.exists(), f'Missing V5 checkpoint: {v5_path}. Run the V5 training section before V6.'

v4_ckpt=torch.load(v4_path, map_location=DEVICE)
v5_ckpt=torch.load(v5_path, map_location=DEVICE)
v4_cfg=v4_ckpt['config']
v5_cfg=v5_ckpt['config']
item_cols=list(dict.fromkeys([*ITEM_ID_FEATURES,*ITEM_CATEGORICAL,*ITEM_NUMERIC]))
# V4 vocabularies must be rebuilt from the same train-side pools used during V4 training.
vocab_item_pool=read_parquet(ADAPTIVE_ROOT/'sequence_examples/train', item_cols, v4_cfg.get('max_item_pool_examples')).drop_duplicates('video_id', keep='last').reset_index(drop=True)
# The serving candidate pool can be validation/test; cold items map to OOV if unseen during V4 training.
item_pool=read_parquet(ADAPTIVE_ROOT/'sequence_examples'/CONFIG.candidate_pool_split, item_cols).drop_duplicates('video_id', keep='last').reset_index(drop=True)
query_frame=read_parquet(ADAPTIVE_ROOT/'sequence_examples/validation', INPUT_COLUMNS, CONFIG.max_replay_queries, ds.field('target_class')=='STRONG_POSITIVE').reset_index(drop=True)
query_frame=sanitize_point_in_time_sequences(query_frame)
numeric=NumericalPreprocessor(Path(CONFIG.base_gold_root)/'transforms/numeric_stats.json')
vocabs=build_vocabs_for_v4(v4_cfg, vocab_item_pool)
vocab_sizes=v4_ckpt.get('vocab_sizes', {k:v.size for k,v in vocabs.items()})
v4=V4TransformerRetrieval(vocab_sizes, v4_cfg).to(DEVICE)
v4.load_state_dict(v4_ckpt['model_state_dict'])
v4.eval()
v5=V5Ranker(v4_cfg['retrieval_dim'], tuple(v5_cfg['hidden_dims']), v5_cfg['dropout']).to(DEVICE)
v5.load_state_dict(v5_ckpt['model_state_dict'])
v5.eval()
print('query/item pool:', query_frame.shape, item_pool.shape)

In [ ]:

def encode_items(frame, batch_size=8192):
    embs=[]; ids=[]
    with torch.no_grad():
        for s in range(0,len(frame),batch_size):
            part=frame.iloc[s:s+batch_size].reset_index(drop=True)
            dummy=part.assign(**minimal_user_stub(query_frame.iloc[0].to_dict(), len(part)))
            emb=v4.encode_item(move(encode_frame(dummy,vocabs,numeric,v4_cfg['sequence_max_len']),DEVICE)).cpu().numpy()
            embs.append(emb); ids.append(part['video_id'].to_numpy('int64'))
    return np.concatenate(ids), np.vstack(embs).astype('float32')

item_ids,item_emb=encode_items(item_pool)
print('item embeddings:', item_emb.shape)

def retrieve_and_rerank(frame):
    rows=[]; k=CONFIG.retrieval_candidates_per_query
    for s in range(0,len(frame),128):
        qb=frame.iloc[s:s+128].reset_index(drop=True)
        with torch.no_grad():
            user_emb=v4.encode_user(move(encode_frame(qb,vocabs,numeric,v4_cfg['sequence_max_len']),DEVICE)).cpu().numpy()
        scores=user_emb @ item_emb.T; kk=min(k, scores.shape[1])
        idx=np.argpartition(-scores, kth=kk-1, axis=1)[:,:kk]
        vals=np.take_along_axis(scores, idx, axis=1); order=np.argsort(-vals, axis=1); idx=np.take_along_axis(idx,order,axis=1); vals=np.take_along_axis(vals,order,axis=1)
        flat_u=[]; flat_i=[]; flat_s=[]
        for qi in range(len(qb)):
            for ci,sc in zip(idx[qi], vals[qi]): flat_u.append(user_emb[qi]); flat_i.append(item_emb[ci]); flat_s.append(sc)
        with torch.no_grad():
            v5_scores=v5(torch.as_tensor(np.vstack(flat_u),device=DEVICE), torch.as_tensor(np.vstack(flat_i),device=DEVICE), torch.as_tensor(flat_s,device=DEVICE,dtype=torch.float32)).cpu().numpy()
        off=0
        for qi,row in qb.iterrows():
            n=idx.shape[1]; vids=item_ids[idx[qi]]; v4_scores=vals[qi]; rerank=v5_scores[off:off+n]; off+=n
            rows.append({'example_id':row['example_id'],'user_id':int(row['user_id']),'true_video_id':int(row['video_id']),'history_len':int(row.get('recent_event_count',0) or 0),'item_hist_events':float(row.get('item_hist_events',0) or 0),'shift_score':abs(float(row.get('session_vs_user_long_view_delta',0) or 0))+abs(float(row.get('session_vs_user_watch_ratio_delta',0) or 0)),'v4_ranked':vids[np.argsort(-v4_scores)].tolist(),'v5_ranked':vids[np.argsort(-rerank)].tolist()})
    return rows

replay_rows=retrieve_and_rerank(query_frame)
print('scored replay rows:', len(replay_rows))


In [ ]:

def metrics(rows, key, top_ks):
    out={}
    for k in top_ks:
        hits=[]; nd=[]
        for r in rows:
            arr=np.asarray(r[key][:k]); pos=np.where(arr==r['true_video_id'])[0]
            hits.append(len(pos)>0); nd.append(0 if len(pos)==0 else 1/math.log2(int(pos[0])+2))
        out[f'hitrate@{k}']=float(np.mean(hits)); out[f'ndcg@{k}']=float(np.mean(nd))
    return out

def segment(rows, predicate):
    return [r for r in rows if predicate(r)]

v4_metrics=metrics(replay_rows,'v4_ranked',CONFIG.top_ks)
v5_metrics=metrics(replay_rows,'v5_ranked',CONFIG.top_ks)
comparison={'v4_adaptive_retrieval':v4_metrics,'v4_plus_v5_ranker':v5_metrics,'ranker_delta_vs_v4':{k:v5_metrics[k]-v4_metrics.get(k,0) for k in v5_metrics},'evaluated_queries':len(replay_rows)}
segments={
    'short_history': segment(replay_rows, lambda r: r['history_len'] < 5),
    'long_history': segment(replay_rows, lambda r: r['history_len'] >= 20),
    'cold_item_proxy': segment(replay_rows, lambda r: r['item_hist_events'] <= 0),
    'warm_item_proxy': segment(replay_rows, lambda r: r['item_hist_events'] > 0),
    'recent_preference_shift_proxy': segment(replay_rows, lambda r: r['shift_score'] >= 0.1),
    'stable_preference_proxy': segment(replay_rows, lambda r: r['shift_score'] < 0.1),
}
comparison['segments']={name:{'queries':len(rows),'v4':metrics(rows,'v4_ranked',CONFIG.top_ks) if rows else {},'v4_v5':metrics(rows,'v5_ranked',CONFIG.top_ks) if rows else {}} for name,rows in segments.items()}
save_json(OUTPUT_DIR/'online_model_comparison.json', comparison)
report_rows = [{k:v for k,v in r.items() if k not in {'v4_ranked','v5_ranked'}} for r in replay_rows[:CONFIG.max_report_rows]]
pd.DataFrame(report_rows).to_csv(OUTPUT_DIR/'online_replay_rows.csv', index=False)
print(f'saved compact replay report rows: {len(report_rows)} / {len(replay_rows)}')
print(json.dumps(comparison, indent=2)[:5000])


In [ ]:
# Metric scaffold: combine saved V3/V4/V5 references and online-state segmentation.
summary={
    'replay_queries': int(len(queries)),
    'users': int(queries['user_id'].nunique()),
    'short_history_queries': int((state_df['history_len'] < 5).sum()),
    'long_history_queries': int((state_df['history_len'] >= 20).sum()),
}
if v3_metrics_path.exists(): summary['v3_reference']=json.loads(v3_metrics_path.read_text())
if v4_metrics_path.exists(): summary['v4_reference']=json.loads(v4_metrics_path.read_text())
if v5_metrics_path.exists(): summary['v5_reference']=json.loads(v5_metrics_path.read_text())
save_json(OUTPUT_DIR/'online_replay_summary.json', summary)
print(json.dumps({k:v for k,v in summary.items() if not k.endswith('_reference')}, indent=2))

## Download Local V6 Artifacts

Artifacts are written to Colab local disk. Run this cell after replay to download them to your machine.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/v6_online_replay', 'zip', str(OUTPUT_DIR))
print('created', zip_path)
files.download(zip_path)